# S&P 500 Data Preparation & Exploratory Data Analysis (EDA)

In [70]:
# Import the needed libraries
import numpy as np 
import pandas as pd 
import yfinance as yf
# Setting Plotly backend plotting for pandas 
pd.options.plotting.backend = 'plotly'
import plotly.io as pio
pio.templates.default = 'plotly_dark'
import plotly.graph_objects as go
import holidays 
from statsmodels.tsa.stattools import adfuller


In [71]:
# import raw data from yfinance
raw = yf.download('^GSPC', start='2000-01-01')

[*********************100%***********************]  1 of 1 completed


In [72]:
# check our loaded data 
raw

Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,
2000-01-03,1455.219971,1478.000000,1438.359985,1469.250000,931800000
2000-01-04,1399.420044,1455.219971,1397.430054,1455.219971,1009000000
2000-01-05,1402.109985,1413.270020,1377.680054,1399.420044,1085500000
2000-01-06,1403.449951,1411.900024,1392.099976,1402.109985,1092300000
2000-01-07,1441.469971,1441.469971,1400.729980,1403.449951,1225200000
...,...,...,...,...,...
2026-05-04,7200.750000,7244.540039,7174.120117,7228.379883,5215480000
2026-05-05,7259.220215,7273.259766,7233.620117,7233.620117,5279760000


In [73]:
# No duplicate index check
# yfinance occasionally returns duplicate dates
assert raw.index.duplicated().sum() == 0, "Duplicate dates found in index"

### 📊 Understanding OHLCV Market Data

Each row in the dataset represents a **single trading day** in the market.


**Open**  
The first recorded price at the start of the trading session (**9:30 AM EST**).  
It often reflects **overnight news, earnings, macroeconomic developments, and shifts in investor sentiment** while markets were closed.


**High & Low**  
The **maximum (High)** and **minimum (Low)** prices reached during the trading session.

- Together, they define the **intraday range**
- This provides insight into **market volatility and uncertainty**
- Larger ranges typically indicate **strong reactions to new information or increased trading activity**


**Close** ⭐ *(Most Important)*  
The final price recorded at the end of the trading session (**4:00 PM EST**).

- Represents the market’s **final consensus on valuation**
- The **primary variable used in financial analysis and modeling**
- Most time series models are built using:
  - Closing prices, or
  - Returns derived from closing prices


**Volume**  
Represents the level of trading activity during the day.

- For indices like the **S&P 500**, volume is **not directly observed**
- It is **derived or provided by the data source**
- As a result:
  - It may be **less reliable**
  - It is **not typically used as a primary feature in modeling**

## Data Inspection 

In [74]:
# Check the DataSet shape
raw.shape

(6627, 5)

In [75]:
type(raw.index)

pandas.DatetimeIndex

In [76]:
# Check for missing values 
raw.isnull().sum()

Price   Ticker
Close   ^GSPC     0
High    ^GSPC     0
Low     ^GSPC     0
Open    ^GSPC     0
Volume  ^GSPC     0
dtype: int64

In [77]:
# Check Data description 
summary = raw.describe()
summary

Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
count,6627.000000,6627.000000,6627.000000,6627.000000,6.627000e+03
mean,2330.243556,2343.092603,2315.625198,2329.858076,3.446917e+09
std,1538.254872,1544.958713,1530.242226,1537.842917,1.526296e+09
min,676.530029,695.270020,666.789978,679.280029,0.000000e+00
25%,1213.540039,1220.304993,1205.494995,1213.125000,2.342885e+09
50%,1552.359985,1556.770020,1545.130005,1552.099976,3.544030e+09
75%,2937.935059,2946.170044,2918.970093,2932.314941,4.292735e+09
max,7398.930176,7401.500000,7362.970215,7376.779785,1.145623e+10


In [78]:
len(raw)

6627

The dataset contains **~6,600+ daily observations**, providing a sufficiently large sample for statistical analysis and modeling.

The minimum value of the **Volume** column is **0**, which is unexpected for a confirmed trading day.

 This warrants further investigation to assess whether it represents a data integrity issue or a legitimate market condition.

In [79]:
# Inspect your current column structure
raw.columns

MultiIndex([( 'Close', '^GSPC'),
            (  'High', '^GSPC'),
            (   'Low', '^GSPC'),
            (  'Open', '^GSPC'),
            ('Volume', '^GSPC')],
           names=['Price', 'Ticker'])

In [80]:
# Flatten multi-level columns

if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
raw = raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

# Validate the dataset flatten
print("Columns after flattening:", raw.columns.tolist())


Columns after flattening: ['Open', 'High', 'Low', 'Close', 'Volume']


In [81]:
raw.head()

Price,Open,High,Low,Close,Volume
Date,,,,,
2000-01-03,1469.250000,1478.000000,1438.359985,1455.219971,931800000
2000-01-04,1455.219971,1455.219971,1397.430054,1399.420044,1009000000
2000-01-05,1399.420044,1413.270020,1377.680054,1402.109985,1085500000
2000-01-06,1402.109985,1411.900024,1392.099976,1403.449951,1092300000
2000-01-07,1403.449951,1441.469971,1400.729980,1441.469971,1225200000


### 🧩 Column Structure Adjustment: Flattening Multi-Level Headers

The dataset initially contains **multi-level column headers** due to the structure returned by the data source (*yfinance*), where each column is paired with the ticker symbol.

While this format is useful when working with **multiple assets**, it introduces unnecessary complexity in a **single-index context**.


**Why this matters**

- Complicates column selection and referencing  
- Increases the risk of errors in:
  - Feature engineering  
  - Data transformations  
  - Model input preparation  


**Action Taken**

- The multi-level columns were **flattened to a single level**
- This ensures:
  - Simpler column access  
  - Cleaner data manipulation  
  - Greater consistency across the pipeline  


**Key Insight**

> Standardizing the dataset structure early improves **robustness, readability, and maintainability** of the data pipeline.

In [82]:
#  Zero Volume Investigation
print("=== ZERO VOLUME DAYS ANALYSIS ===")
zero_volume = raw[raw['Volume'] == 0]
print(f"Total days with zero volume: {len(zero_volume)}")
print(f"Percentage of dataset: {len(zero_volume)/len(raw)*100:.4f}%")

=== ZERO VOLUME DAYS ANALYSIS ===
Total days with zero volume: 1
Percentage of dataset: 0.0151%


In [83]:
# Context around that day
if len(zero_volume) > 0:
    idx = zero_volume.index[0]
    window = raw.loc[idx - pd.Timedelta(days=5): idx + pd.Timedelta(days=5)]
    print("\n=== 5 Days Before & After Context ===")
    display(window[['Close', 'Volume']])


=== 5 Days Before & After Context ===


Price,Close,Volume
Date,,
2023-05-19,4191.979980,4041900000
2023-05-22,4192.629883,3728520000
2023-05-23,4145.580078,4155320000
2023-05-24,4115.240234,0
2023-05-25,4151.279785,4147760000
2023-05-26,4205.450195,3715460000


In [84]:
# programmatically verified zero-volume days against US holiday calendars
us_holidays = holidays.US()

zero_vol_dates = raw[raw['Volume'] == 0].index
for date in zero_vol_dates:
    holiday = us_holidays.get(date)
    print(f"{date.date()} | Holiday: {holiday if holiday else 'None — anomalous'}")

2023-05-24 | Holiday: None — anomalous


### ⚠️ Data Integrity Check: Zero-Volume Observation

During data validation, a **zero-volume entry** was identified on:

- **Date:** May 24, 2023


**Verification**

- Cross-checked against US market holidays using the `holidays` library  
- Confirmed: **Not a market holiday** → valid trading day  


**Assessment**

- The **S&P 500 is an index**, not a directly traded asset  
- Volume is:
  - **Derived / synthetic**
  - Dependent on the data provider (*yfinance*)  
- A zero value on a valid trading day indicates a **data integrity issue**, not a real market condition  


**Action Taken**

- The observation was **removed from the dataset**
- **Rationale**:
  - A zero-volume value on a confirmed trading day indicates a data integrity issue  
  - Removing the row prevents introducing artificial values or distortions  
  - Ensures a clean and reliable foundation for downstream analysis and modeling  


**Key Insight**

> External financial data sources must be **validated**, not blindly trusted.

In [85]:
# S&P 500 Volume Anomaly Detection Plot


# Define anomaly date
anomaly_date = '2023-05-24'
anomaly_volume = raw.loc[anomaly_date, 'Volume']

# Create figure
fig = go.Figure()

# Main volume line
fig.add_trace(
    go.Scatter(
        x=raw.index,
        y=raw['Volume'],
        mode='lines',
        name='Volume'
    )
)

fig.add_trace(
    go.Scatter(
        x=[anomaly_date],
        y=[anomaly_volume],
        mode='markers',
        name='Anomaly',
        marker=dict(size=12)
    )

)

fig.add_annotation(
    x=anomaly_date,
    y=anomaly_volume,
    text=(
        "Zero volume recorded on a confirmed trading day"
        "<br>→ Potential provider inconsistency"
    ),
    showarrow=True,
    arrowhead=2,
    ax=120,
    ay=-80,
    font=dict(size=12)

)

# Layout
fig.update_layout(
    title='S&P 500 Volume Anomaly Detection',
    xaxis_title='Date',
    yaxis_title='Volume',
    template='plotly_dark',
    height=700,
    width=1400,
    hovermode='x unified'
)

# Show figure
fig.show()
fig.write_image(
    "../reports/figures/sp500_volume_anomaly.png",
    width=1200, 
    height=627, 
    scale=2
)

In [86]:
# We drop the rows that has Zero volume by 
raw = raw[raw['Volume']  > 0]

In [87]:
# Confirm no zero-volume rows remain
raw[raw['Volume'] == 0]

Price,Open,High,Low,Close,Volume
Date,,,,,


The latest observation may represent **partial or delayed data**, depending on when the API request is made.

To ensure data quality, the final row is removed to prevent any incomplete or partially recorded observations from entering the pipeline.

This approach avoids introducing bias into:
- **Return calculations**
- **Volatility estimates**

In [88]:
# Enforce use of fully confirmed data only (T-1)
raw = raw.iloc[:-1]


In [89]:
# Confirm the last row
raw.tail()

Price,Open,High,Low,Close,Volume
Date,,,,,
2026-05-01,7234.540039,7272.520020,7229.319824,7230.120117,4847390000
2026-05-04,7228.379883,7244.540039,7174.120117,7200.750000,5215480000
2026-05-05,7233.620117,7273.259766,7233.620117,7259.220215,5279760000
2026-05-06,7294.140137,7369.220215,7294.140137,7365.120117,6579640000
2026-05-07,7376.779785,7385.020020,7321.250000,7337.109863,6237040000


In [90]:
# Visualize Close Price over time
fig = raw['Close'].plot(title='S&P 500 Close Price (2000–Present)')
fig.update_layout(
    showlegend=False,
    yaxis_title='S&P 500',
    xaxis_title='Date'
)
fig.show()

To assess suitability for **financial time series modeling**, raw closing prices are transformed into **log returns**.

Raw prices are typically **non-stationary**, exhibiting trends and compounding effects that violate the assumptions of many statistical models. Transforming prices into log returns removes these effects and results in a series with more stable statistical properties.

Log returns are widely used in quantitative finance due to their **additivity over time** and suitability for modeling relative price changes.

The resulting series is then analyzed to evaluate its statistical behavior prior to model selection.

In [91]:
# We create a new column and calculate the daily log Returns

raw['log_returns'] = np.log(raw['Close']/ raw['Close'].shift(1))



In [92]:
# Double check our results

raw[['Close', 'log_returns']].head(10)

Price,Close,log_returns
Date,,
2000-01-03,1455.219971,NaN
2000-01-04,1399.420044,-0.039099
2000-01-05,1402.109985,0.001920
2000-01-06,1403.449951,0.000955
2000-01-07,1441.469971,0.026730
2000-01-10,1457.599976,0.011128
2000-01-11,1438.560059,-0.013149
2000-01-12,1432.250000,-0.004396
2000-01-13,1449.680054,0.012096


In [93]:
# I drop the first row as we can't compute daily returns for the first day of our data 
raw = raw.dropna(subset=['log_returns'])

# To confirm that all NaN values dropped:
raw['log_returns'].isna().sum()

np.int64(0)

In [94]:
raw['log_returns'].describe()

count    6624.000000
mean        0.000244
std         0.012177
min        -0.127652
25%        -0.004732
50%         0.000639
75%         0.005880
max         0.109572
Name: log_returns, dtype: float64

The mean of log returns is close to zero, which is consistent with empirical findings in financial markets. However, this alone does not imply predictability or stationarity.

However, this alone does **not confirm stationarity**. Formal statistical tests (e.g., Augmented Dickey-Fuller) are required to assess whether the series satisfies stationarity assumptions.

The standard deviation of log returns represents **daily volatility**, indicating the typical magnitude of price fluctuations. In this dataset, it is approximately **1.2% per day**, which is consistent with historical equity market behavior.

In [95]:
# We check the min and max dates for the daily returns :
# Find the actual dates of extreme moves
raw['log_returns'].nsmallest(5)

Date
2020-03-16   -0.127652
2020-03-12   -0.099945
2008-10-15   -0.094695
2008-12-01   -0.093537
2008-09-29   -0.092190
Name: log_returns, dtype: float64

In [96]:
raw['log_returns'].nlargest(5)

Date
2008-10-13    0.109572
2008-10-28    0.102457
2025-04-09    0.090895
2020-03-24    0.089683
2020-03-13    0.088808
Name: log_returns, dtype: float64

### 📉 Extreme Market Movements

To better understand tail behavior, the largest positive and negative daily returns were examined.

These extreme observations **exceed typical daily movements by multiple standard deviations**, indicating periods of significant market stress.


**Largest Gains**

- **2008 Financial Crisis** — Multiple large upward moves driven by bailout announcements and relief rallies  
- **March 2020 (COVID-19)** — Sharp rebounds during periods of extreme volatility  
- **April 2025** — Elevated volatility linked to major macroeconomic policy announcements  


**Largest Losses**

- **March 16, 2020** — Largest single-day drop (~-12.7%), following global COVID-19 lockdown escalation  
- **March 12, 2020** — WHO pandemic declaration triggered sharp market decline  
- **2008 Financial Crisis** — Sustained drawdowns during systemic financial instability  


These observations highlight the presence of **fat tails** and **extreme risk events**, which are key characteristics of financial return series.

In [97]:
# Visualize log returns over time
fig = raw['log_returns'].plot(title='S&P 500 Daily Log Returns (2000–Present)')
fig.update_layout(
    showlegend=False,
    yaxis_title='Log Return',
    xaxis_title='Date'
)
fig.show()


### 📈 Volatility Clustering

The log return time series exhibits **volatility clustering**, where periods of high volatility are followed by further high volatility, and periods of relative calm tend to persist.


**Notable High-Volatility Periods**

- **2008 Financial Crisis**  
- **2020 COVID-19 market shock**  
- Elevated volatility observed in **2025–2026**


**Low-Volatility Regime**

- The period between **2013–2019** shows relatively stable and lower volatility  


**Interpretation**

Periods of elevated volatility are characterized by larger absolute returns, indicating increased market uncertainty and risk.

This behavior reflects a key stylized fact of financial markets and indicates that volatility is **time-dependent rather than constant**.

As a result, the assumption of constant variance is violated, which has important implications for modeling. It motivates the use of models that can account for time-varying volatility, such as rolling statistics, GARCH-type models, or more flexible machine learning approaches.

To further quantify this behavior, rolling measures of volatility are computed in the next step.

In [98]:
extremes = raw['log_returns'].nsmallest(1)
extreme_loss = extremes.iloc[0]

extremes_high = raw['log_returns'].nlargest(1)
extreme_gain = extremes_high.iloc[0]

In [99]:
fig = raw['log_returns'].plot(title='S&P 500 Daily Log Returns (2000–Present)', kind='hist')
fig.update_layout(
    showlegend=False,
    yaxis_title='Frequency',
    xaxis_title='Log Return'
)
# Add annotation for the March 2020 COVID Crash
fig.add_annotation(
    x=extreme_loss,             # The ~-12.7% log return identified in your analysis
    y=raw['log_returns'].value_counts().max() * 0.01,                  
    text="March 2020 (COVID-19 Shock)",
    showarrow=True,
    arrowhead=1,
    ax=-40,               # Horizontal offset for the text box
    ay=-40                # Vertical offset for the text box
)
# Add annotation for the 2008 Financial Crisis Relief Rally
fig.add_annotation(
    x=extreme_gain,              # Use the exact value from your nlargest(5) results
    y=raw['log_returns'].value_counts().max() * 0.01,                  # Keep y low on the Frequency axis for the tail
    text="2008 Financial Crisis (Relief Rally)",
    showarrow=True,
    arrowhead=1,
    ax=40,                # Positive offset to place text to the right of the arrow
    ay=-40
)
fig.show()

### 📊 Distribution of Log Returns

The histogram of log returns provides insight into the **distribution of daily market movements**.


**Key Observations**

- The distribution is centered around **zero**, consistent with financial return series  
- The presence of **extreme values** on both sides indicates **fat tails**  
- These tail events correspond to periods of significant market stress  


**Interpretation**

Unlike a normal distribution, financial returns exhibit **higher probability of extreme events**, which is a well-documented characteristic in quantitative finance.

This has important implications for risk modeling, as standard Gaussian assumptions may underestimate tail risk.

The modeling approach focuses on capturing **patterns in return magnitude and volatility**, rather than explaining the underlying economic causes of these movements.

In practice, financial models are typically built on **returns rather than raw prices**, as returns exhibit more stable statistical properties and are more suitable for time series analysis.

# Stationarity Test
##  Augmented Dickey-Fuller (ADF) test

In [100]:
def adf_test(series, name='Series'):
    result = adfuller(series.dropna())
    
    print(f'ADF Test for {name}')
    print('-' * 30)
    print(f'ADF Statistic : {result[0]:.4f}')
    print(f'P-Value       : {result[1]:.6f}')
    if result[1] < 0.05:
        print("✅ Reject null hypothesis — series is stationary")
    else:
        print('Fail to Reject null hypothesis - series is Not Stationary')
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'   {key}: {value:.4f}')




In [101]:
adf_test(raw['log_returns'], 'Log Returns')

ADF Test for Log Returns
------------------------------
ADF Statistic : -19.5882
P-Value       : 0.000000
✅ Reject null hypothesis — series is stationary
Critical Values:
   1%: -3.4313
   5%: -2.8620
   10%: -2.5670


### 📊 Stationarity Test: Augmented Dickey-Fuller (ADF)

To formally assess whether the series is suitable for time series modeling, the **log return series** was tested using the **Augmented Dickey-Fuller (ADF) test**.


**Test Results**

- **ADF Statistic:** -19.60  
- **P-Value:** < 0.001  
- **Critical Values:**
  - 1%: -3.43  
  - 5%: -2.86  
  - 10%: -2.57  


**Interpretation**

- The **p-value is significantly below 0.05**, allowing us to reject the null hypothesis of a unit root  
- The **ADF statistic (-19.60)** is substantially more negative than all critical values  

Together, these results provide strong statistical evidence that the **log return series is stationary**.


**Conclusion**

While raw S&P 500 prices are typically non-stationary due to trends and compounding effects, transforming them into **log returns** produces a series with statistical properties suitable for time series modeling.


**Implications for Modeling**

- Many classical models (e.g., **ARIMA, SARIMA**) assume stationarity  
- Using a stationary series reduces the risk of **spurious relationships**  
- Establishing this property ensures a more reliable foundation for downstream modeling

# Rolling Volatility 

In [102]:
raw['rolling_vol_30'] = raw['log_returns'].rolling(window=30).std()

fig = raw['rolling_vol_30'].plot(title='30-Day Rolling Volatility (S&P 500)')
fig.update_layout(
    showlegend=False,
    yaxis_title='Volatility',
    xaxis_title='Date'
)
fig.show()

### 📊 Rolling Volatility (30-Day)

To quantify time-varying volatility, a 30-day rolling standard deviation of log returns is computed.


**Observations**

- Volatility spikes align with major market events:
  - 2008 Financial Crisis  
  - 2020 COVID-19 shock  
- Periods such as 2013–2019 show sustained low volatility  


**Interpretation**

This confirms that volatility is not constant over time and reinforces the presence of **volatility clustering**.

Rolling volatility provides a simple but effective way to capture changing market risk over time.

In [103]:
# === DRAWDOWN ANALYSIS (High Value for Risk Roles) ===
raw['Return'] = raw['Close'].pct_change()                    # ensure Return column exists
raw['Cumulative'] = (1 + raw['Return']).cumprod()
raw['Peak'] = raw['Cumulative'].cummax()
raw['Drawdown'] = (raw['Cumulative'] - raw['Peak']) / raw['Peak']

print(f"Maximum Drawdown: {raw['Drawdown'].min():.2%}")
print(f"Date of Max Drawdown: {raw['Drawdown'].idxmin().date()}")
print(f"Current Drawdown (as of last date): {raw['Drawdown'].iloc[-1]:.2%}")

# Visualization
fig = raw['Drawdown'].plot(title='S&P 500 Historical Drawdowns')
fig.update_layout(
    yaxis_title='Drawdown (%)',
    xaxis_title='Date',
    showlegend=False
)
fig.show()

# Top 5 worst drawdowns
print("\n=== Top 5 Largest Drawdowns ===")
print(raw['Drawdown'].nsmallest(5))

Maximum Drawdown: -56.78%
Date of Max Drawdown: 2009-03-09
Current Drawdown (as of last date): -0.38%



=== Top 5 Largest Drawdowns ===
Date
2009-03-09   -0.567754
2009-03-05   -0.563908
2009-03-06   -0.563377
2009-03-03   -0.555103
2009-03-02   -0.552235
Name: Drawdown, dtype: float64


### 📉 Drawdown Analysis — Understanding Portfolio Risk

Drawdown measures the **peak-to-trough decline** in cumulative returns — a critical risk metric in finance.

**Key Statistics:**
- **Maximum Drawdown:** -56.78% on **2009-03-09**
- This occurred during the depths of the **2008–2009 Global Financial Crisis**

**Interpretation:**
- A -56.78% drawdown means an investor who entered at the peak would have lost over half their capital at the trough.
- Recovery from such events often takes years, highlighting the importance of risk management.

**Business Relevance:**
- Drawdown analysis is essential for **risk management**, **portfolio construction**, and **stress testing**.
- Understanding historical worst-case scenarios helps quantify tail risk and informs decisions on position sizing, hedging, and stop-loss strategies.
- Future anomaly detection and modeling phases will build on this foundation.

**Visualization:** See the interactive plot above showing drawdown evolution over 26+ years.

In [104]:
# === CALENDAR & SEASONALITY ANALYSIS ===
raw['DayOfWeek'] = raw.index.day_name()
raw['Month'] = raw.index.month_name()
raw['Year'] = raw.index.year

# Average return by day of week
dow_returns = raw.groupby('DayOfWeek')['log_returns'].mean().sort_values()
print("Average Log Return by Day of Week:\n", dow_returns)

# Average return by month
month_returns = raw.groupby('Month')['log_returns'].mean().sort_values()
print("\nAverage Log Return by Month:\n", month_returns)

# Plot
import plotly.express as px
fig = px.bar(dow_returns.reset_index(), x='DayOfWeek', y='log_returns', 
             title='Average Daily Log Return by Day of Week')
fig.show()

Average Log Return by Day of Week:
 DayOfWeek
Friday       0.000011
Monday       0.000116
Thursday     0.000207
Wednesday    0.000336
Tuesday      0.000534
Name: log_returns, dtype: float64

Average Log Return by Month:
 Month
September   -0.000714
February    -0.000315
June        -0.000060
January     -0.000037
August       0.000044
December     0.000247
May          0.000310
March        0.000328
October      0.000534
July         0.000693
April        0.000849
November     0.000984
Name: log_returns, dtype: float64


### 📅 Calendar & Seasonality Analysis

Financial markets exhibit subtle but persistent seasonal patterns. We analyzed average log returns by day-of-week and month.

**Average Log Return by Day of Week:**
- **Strongest:** Tuesday (0.000528), Wednesday (0.000326)
- **Weakest:** Friday (0.000011)

**Average Log Return by Month:**
- **Strongest:** November (0.000984), April (0.000849), July (0.000693)
- **Weakest:** September (-0.000714), February (-0.000315)

**Interpretation:**
- The weak Friday performance may reflect traders closing positions ahead of the weekend.
- Strong November performance is consistent with year-end rally / “Santa Claus Rally” behavior.
- September weakness is a well-documented historical phenomenon (September historically exhibits weaker average returns relative to other months.”).

**Business Relevance:**
Understanding these patterns supports tactical allocation, options strategy timing, and feature engineering (e.g., day-of-week dummy variables). While not strong enough for standalone trading signals, they are valuable regime-detection features.

In [107]:
# === SAVE CLEANED DATASET FOR FUTURE PHASES ===
import os
os.makedirs('data', exist_ok=True)

# Save both raw (with all features) and a minimal version
raw.to_csv('../data/sp500_cleaned.csv')
raw[['Open', 'High', 'Low', 'Close', 'Volume', 'log_returns']].to_csv('../data/sp500_features.csv')

print("✅ Cleaned datasets saved to ./data/")
print(f"Final shape: {raw.shape}")

✅ Cleaned datasets saved to ./data/
Final shape: (6624, 14)


### ✅ EDA Summary

Key findings from the exploratory analysis:

- **Data Integrity**  
  - Identified and removed anomalous zero-volume observation  
  - Ensured clean and consistent dataset  

- **Return Transformation**  
  - Prices transformed into log returns for modeling suitability  

- **Statistical Properties**  
  - Log returns exhibit near-zero mean and time-varying volatility
  - ADF test confirms stationarity  

- **Market Behavior**  
  - Presence of **volatility clustering**  
  - Evidence of **fat tails and extreme events**  
  - Time-varying volatility confirmed through rolling measures  

---

These findings establish a strong and statistically sound foundation for the next phase: **feature engineering and predictive modeling**.